# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Toàn bộ quy trình từ tải dữ liệu, chuẩn bị nhãn, huấn luyện SFT (E0-E3), ablation (E4) và GRPO (E5) được hiện thực đầy đủ bên dưới.

## 0. Môi trường (Colab T4)

In [ ]:
# 0. Điều hướng vào thư mục dự án (hoặc clone nếu đang ở /content)
import os
if not os.path.exists('scripts'):
    if not os.path.exists('K4-Track-4-Day10-VisionLanguageModel'):
        !git clone https://github.com/ngmai2005/K4-Track-4-Day10-VisionLanguageModel.git
    %cd K4-Track-4-Day10-VisionLanguageModel

# 1. Kiểm tra phần cứng GPU NVIDIA T4 16GB
!nvidia-smi -L

# 2. Cài đặt các thư viện phụ thuộc theo phiên bản chuẩn
!pip -q install -r requirements-colab.txt

# 3. Chạy toàn bộ unit tests kiểm tra tính toàn vẹn của repo đề
!python -m pytest -q

In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

## 1. Dữ liệu: tải và chia split

In [ ]:
# Tải dataset Road Damage từ Kaggle và chia theo session cố định
# Sinh ra: data/images/, data/splits/{train,val,test}.jsonl và data/splits/meta.json
!python scripts/prepare_data.py --kaggle

In [ ]:
# Kiểm tra số lượng mẫu và trực quan hoá 6 ảnh train kèm bounding box gốc
import json
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from scripts.labcommon import read_jsonl

train_samples = read_jsonl('data/splits/train.jsonl')
val_samples = read_jsonl('data/splits/val.jsonl')
test_samples = read_jsonl('data/splits/test.jsonl')

print(f'Train: {len(train_samples)} ảnh | Val: {len(val_samples)} ảnh | Test: {len(test_samples)} ảnh')

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
color_map = {'pothole': '#e74c3c', 'crack': '#f39c12', 'manhole': '#3498db'}
for idx, ax in enumerate(axes.flatten()):
    entry = train_samples[idx]
    img = Image.open(f"data/{entry['image_path']}").convert('RGB')
    draw = ImageDraw.Draw(img)
    w, h = entry['width'], entry['height']
    for obj in entry['objects']:
        xc, yc, bw, bh = obj['yolo']
        x1, y1 = (xc - bw / 2) * w, (yc - bh / 2) * h
        x2, y2 = (xc + bw / 2) * w, (yc + bh / 2) * h
        color = color_map.get(obj['label'], 'lime')
        draw.rectangle([x1, y1, x2, y2], outline=color, width=2)
        draw.text((x1 + 2, max(0, y1 - 12)), obj['label'], fill=color)
    ax.imshow(img)
    ax.set_title(f"{entry['session']}\n{entry['image_id'][:16]}", fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
# Sử dụng hàm chuyển đổi nhãn chuẩn hoá thang 0-1000 và sắp xếp (y1, x1) từ lab_helpers.py
from lab_helpers import PROMPT, yolo_to_box, objects_to_answer

sample_entry = train_samples[0]
sample_json = objects_to_answer(sample_entry['objects'])
print('--- PROMPT ---')
print(PROMPT)
print('\n--- GROUNDING JSON TARGET (thang 0-1000, sort y1, x1) ---')
print(sample_json)

# Kiểm tra tính nhất quán chuyển đổi ngược lại
parsed_boxes = json.loads(sample_json)
for b in parsed_boxes:
    print(f"  Lớp: {b['label']:10s} BBox 2D: {b['bbox_2d']}")

## 3. E0: zero-shot

In [ ]:
# Chạy suy luận zero-shot mô hình gốc Qwen3-VL-4B-Instruct trên 311 ảnh test
!python run_lab.py --experiment E0

In [ ]:
# Chấm điểm E0 bằng trọng tài score.py
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

# Hiển thị tóm tắt kết quả E0
with open('report/evidence/e0.json', 'r', encoding='utf-8') as f:
    e0_data = json.load(f)
print(f"E0 Parse Valid Rate : {e0_data['parse_valid_rate']:.4f}")
print(f"E0 Micro F1          : {e0_data['micro']['f1']:.4f}")

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# Kiểm tra chat template, token ảnh (220 tokens) và mask loss (-100)
import torch
from transformers import AutoProcessor
from vl_training import encode_example

processor = AutoProcessor.from_pretrained('Qwen/Qwen3-VL-4B-Instruct')
processor.tokenizer.padding_side = 'left'

sample = train_samples[0]
sample_img = Image.open(f"data/{sample['image_path']}").convert('RGB')
sample_ans = objects_to_answer(sample['objects'])
encoded = encode_example(sample_img, PROMPT, sample_ans, processor)

# Đếm token thị giác: ảnh 640x360 qua patch 14x14 + merge 2x2 = đúng 220 visual tokens
grid = encoded['image_grid_thw']
vision_tokens = int(grid[0, 1] * grid[0, 2] // 4)
print(f'image_grid_thw : {grid.tolist()}')
print(f'Vision tokens  : {vision_tokens} (chính xác 220 tokens)')

# Giải mã ngược token có label != -100: phải ra đúng chuỗi JSON đáp án
assistant_ids = encoded['labels'][encoded['labels'] != -100]
decoded_ans = processor.tokenizer.decode(assistant_ids, skip_special_tokens=False)
print(f'Decoded labels != -100:\n{decoded_ans}')

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# Phân tích số tham số huấn luyện giữa các setup:
# E1: LoRA LLM (252 modules) | E2: + Projector float32 | E3: + LoRA ViT (96 modules, tổng 348)
from run_lab import load_config

for exp in ['E1', 'E2', 'E3']:
    cfg = load_config(exp)
    print(f"[{exp}] targets={cfg['lora']['targets']} | train_projector={cfg.get('train_projector')}")

## 6. Vòng huấn luyện

In [ ]:
# Huấn luyện tuần tự E1, E2, E3 (300 bước tối ưu, grad accum 4, batch size 2, AdamW lr 2e-4)
# Mỗi setup tự động lưu checkpoint adapter và projector (E2/E3) trong work/<exp>/adapter
!python run_lab.py --experiment E1
!python run_lab.py --experiment E2
!python run_lab.py --experiment E3

## 7. Suy luận và chấm E1–E3

In [ ]:
# Hiển thị bảng tổng hợp số liệu E0 - E3 từ các file evidence đã sinh
from pathlib import Path

print(f"{'Setup':5s} | {'F1':6s} | {'CI95':15s} | {'VRAM':7s} | {'Phút':6s} | {'Trainable':10s}")
print('-' * 60)
for exp in ['e0', 'e1', 'e2', 'e3']:
    p = Path(f'report/evidence/{exp}.json')
    if p.exists():
        d = json.loads(p.read_text(encoding='utf-8'))
        f1 = d['micro']['f1']
        ci = f"[{d['ci95'][0]:.3f}, {d['ci95'][1]:.3f}]"
        vram = f"{d['meta'].get('peak_vram_gb', 0):.2f}G" if 'meta' in d and 'peak_vram_gb' in d['meta'] else 'N/A'
        mins = f"{d['meta'].get('train_minutes', 0):.1f}" if 'meta' in d and 'train_minutes' in d['meta'] else 'N/A'
        trainable = f"{d['meta'].get('trainable_params', 0):,}" if 'meta' in d and 'trainable_params' in d['meta'] else '0'
        print(f"{exp.upper():5s} | {f1:.4f} | {ci:15s} | {vram:7s} | {mins:6s} | {trainable:10s}")

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# 1. Chạy Ablation Study E4 trên biến rank: r=4 và r=64
!python run_lab.py --experiment E4 --rank 4
!python run_lab.py --experiment E4 --rank 64

# 2. Chạy unit tests cho hàm Verifiable IoU Reward (Bonus E5)
!python -m pytest -v test_e5_reward.py

# 3. Huấn luyện GRPO E5 (+10 điểm thưởng)
!python run_lab.py --experiment E5

# 4. Vẽ tối thiểu 6 ảnh minh hoạ dự đoán và ground truth
!python render_examples.py --pred work/e1.jsonl --count 6 --out report/examples

# 5. Kiểm tra tự động điều kiện nghiệm thu trước khi nộp bài
!python scripts/check_submission.py